# Limpieza reproducible · cuaderno del proyecto (sesión 5)
**Estud-IA · Análisis de Datos Básico**

Este cuaderno **parte siempre del original** y aplica tus reglas de limpieza una por una. Cada regla anota sola cuántas filas afecta. Al final, los conteos tienen que cuadrar: **original = limpias + quitadas + a revisión**.

Corre las celdas en orden con ▶ (o Shift + Enter). Si Colab avisa que el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

**Tres reglas de oro**
1. El archivo de `datos/originales/` **no se toca**. Todo sale de aquí, con código.
2. Una fila que no sabes cómo arreglar **no se inventa ni se borra en silencio**: va **a revisión**.
3. Si te equivocas o cambias una regla: menú **Entorno de ejecución › Reiniciar y ejecutar todo**. Tiene que dar lo mismo cada vez.

Con el enlace que viene (la papelería sucia, 11 filas) debe quedar: **8 limpias, 1 quitada y 2 a revisión**.

In [1]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta, hoja=0):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta, sheet_name=hoja)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas (original, base, analisis) y muestra el resultado."""
    resultado = duckdb.sql(consulta).df()
    for col in resultado.select_dtypes("float").columns:
        if resultado[col].dropna().mod(1).eq(0).all():
            resultado[col] = resultado[col].astype("Int64")   # 45000 en lugar de 45000.0
    return resultado

def _filas(n):
    return f"{n} fila" if n == 1 else f"{n} filas"

def perfil(tabla):
    """Cuántas filas y columnas hay, cuántas filas repetidas y cuántos vacíos tiene cada columna."""
    print(f"{_filas(len(tabla))} · {len(tabla.columns)} columnas · filas idénticas a otra: {int(tabla.duplicated().sum())}")
    return pd.DataFrame({
        "tipo": [str(t) for t in tabla.dtypes],
        "vacíos": tabla.isna().sum(),
        "valores distintos": tabla.nunique(),
        "ejemplo": [tabla[c].dropna().iloc[0] if tabla[c].notna().any() else "" for c in tabla.columns],
    })

def empezar(tabla):
    """Copia el original y deja la bitácora en cero. Todo vuelve a empezar desde aquí."""
    global base, quitadas, revision, bitacora, _original
    _original = tabla.copy()
    base = tabla.copy()
    quitadas = tabla.iloc[0:0].copy()
    revision = tabla.iloc[0:0].copy()
    bitacora = []
    print(f"Empezamos desde el original: {len(base)} filas.")

def _ya_esta(nombre):
    if any(b["regla"] == nombre for b in bitacora):
        raise RuntimeError(f"La regla «{nombre}» ya se aplicó. Corre otra vez desde la celda 4 (Empezar) para no aplicarla dos veces.")

def regla(nombre, filas, motivo, accion="quitar"):
    """Aplica una regla a las filas marcadas con True.
    accion = "quitar": salen de la base (por ejemplo, copias exactas).
    accion = "revisar": salen de la base y van al archivo de revisión (no se inventan ni se pierden)."""
    global base, quitadas, revision
    _ya_esta(nombre)
    filas = pd.Series(filas, index=base.index).fillna(False).astype(bool)
    n, antes = int(filas.sum()), len(base)
    afectadas = _original.loc[base.index[filas]].copy()   # con los valores originales, para poder revisarlas
    afectadas["regla"] = nombre
    if accion == "quitar":
        quitadas = pd.concat([quitadas, afectadas])
    elif accion == "revisar":
        revision = pd.concat([revision, afectadas])
    else:
        raise ValueError('accion debe ser "quitar" o "revisar"')
    base = base[~filas].copy()
    bitacora.append({"regla": nombre, "filas afectadas": n, "acción": accion, "motivo": motivo, "antes": antes, "después": len(base)})
    print(f"{nombre}: {_filas(n)} ({accion}). Antes {antes}, ahora {len(base)}.")
    return afectadas.head(10)

def unificar(columna, motivo="El mismo valor estaba escrito de varias formas"):
    """Quita espacios sobrantes y une las formas que solo cambian en mayúsculas: deja la forma más común."""
    nombre = f"Unificar escritura de {columna}"
    _ya_esta(nombre)
    antes = base[columna].copy()
    texto = antes.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    clave = texto.str.lower()
    mas_comun = texto.groupby(clave).agg(lambda s: s.value_counts().index[0])
    nuevo = clave.map(mas_comun)
    nuevo = nuevo.where(antes.notna(), antes)
    cambiadas = (antes.astype("string") != nuevo.astype("string")).fillna(False)
    base[columna] = nuevo
    bitacora.append({"regla": nombre, "filas afectadas": int(cambiadas.sum()), "acción": "corregir", "motivo": motivo, "antes": len(base), "después": len(base)})
    print(f"{nombre}: filas corregidas: {int(cambiadas.sum())}. Grupos antes: {antes.nunique()}, después: {base[columna].nunique()}.")
    return pd.DataFrame({"antes": antes[cambiadas], "después": nuevo[cambiadas]}).drop_duplicates().head(10)

def reemplazar(columna, cambios, motivo="El mismo valor estaba escrito de otra forma"):
    """Cambia valores exactos por otros: reemplazar("municipio", {"Estrella": "La Estrella"}).
    Sirve cuando unificar() no alcanza: tildes, abreviaturas o nombres distintos de la misma cosa."""
    nombre = f"Reemplazar en {columna}: " + ", ".join(f"{a} -> {b}" for a, b in cambios.items())
    _ya_esta(nombre)
    antes = base[columna].copy()
    base[columna] = base[columna].replace(cambios)
    cambiadas = (antes.astype("string") != base[columna].astype("string")).fillna(False)
    bitacora.append({"regla": nombre, "filas afectadas": int(cambiadas.sum()), "acción": "corregir", "motivo": motivo, "antes": len(base), "después": len(base)})
    print(f"{nombre}: filas corregidas: {int(cambiadas.sum())}. Valores distintos antes: {antes.nunique()}, después: {base[columna].nunique()}.")
    return pd.DataFrame({"antes": antes[cambiadas], "después": base[columna][cambiadas]}).value_counts().rename("filas").reset_index()

def a_numero(columna, miles=None):
    """Convierte una columna a número. Si los miles vienen con punto o coma, escribe miles="." o miles=",".
    Devuelve True en las filas que NO se pudieron convertir (y no estaban vacías): decide qué hacer con ellas."""
    texto = base[columna].astype("string").str.strip()
    if miles:
        texto = texto.str.replace(miles, "", regex=False)
    numero = pd.to_numeric(texto, errors="coerce")
    malas = numero.isna() & base[columna].notna()
    if numero.dropna().mod(1).eq(0).all():
        numero = numero.astype("Int64")   # enteros sin decimales
    base[columna] = numero
    print(f"{columna} ahora es número. Filas que no se pudieron convertir: {int(malas.sum())}.")
    return malas

def a_fecha(columna, formato):
    """Convierte una columna a fecha con un formato explícito (por ejemplo "%Y-%m-%d").
    Devuelve True en las filas que NO se pudieron convertir (y no estaban vacías)."""
    fecha = pd.to_datetime(base[columna].astype("string").str.strip(), format=formato, errors="coerce")
    malas = fecha.isna() & base[columna].notna()
    base[columna] = fecha
    print(f"{columna} ahora es fecha. Filas que no se pudieron leer con el formato {formato}: {int(malas.sum())}.")
    return malas

def ver_bitacora():
    """La bitácora de limpieza como tabla y como líneas para pegar en bitacora.md."""
    tabla = pd.DataFrame(bitacora)
    for b in bitacora:
        print(f"- {b['regla']} · {_filas(b['filas afectadas'])} · {b['acción']} · {b['motivo']}")
    return tabla

def conciliar(original):
    """Comprueba que no se perdió ninguna fila: original = limpias + quitadas + a revisión."""
    total = len(base) + len(quitadas) + len(revision)
    print(f"Original {len(original)} = limpias {len(base)} + quitadas {len(quitadas)} + a revisión {len(revision)}  →  {total}")
    print("Cuadra." if total == len(original) else "NO CUADRA: alguna fila se perdió o se duplicó. Corre todo desde la celda 4.")

def guardar(nombre):
    """Guarda la base limpia y el archivo de revisión como CSV y los descarga a tu computador."""
    base.to_csv(f"{nombre}_limpio.csv", index=False)
    revision.to_csv(f"{nombre}_revision.csv", index=False)
    print(f"Guardé {nombre}_limpio.csv ({len(base)} filas) y {nombre}_revision.csv ({len(revision)} filas).")
    try:
        from google.colab import files
        files.download(f"{nombre}_limpio.csv")
        files.download(f"{nombre}_revision.csv")
    except Exception:
        print("Fuera de Colab: los archivos quedaron en la carpeta del cuaderno.")

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


## 1. Carga el ORIGINAL

Pega el enlace de tu archivo de **`datos/originales/`** (la página de GitHub o el botón **Raw**). Si es un Excel con varias hojas, escribe el nombre de la hoja en `HOJA`.

Con la papelería sucia: **11 filas y 6 columnas**.

In [2]:
# Celda 2 · Cargar el original (no se modifica)
ENLACE = "https://raw.githubusercontent.com/sebas9822-beep/Grupo_E9_Empresas_Exportando/refs/heads/main/datos/originales/muestra_dian_exportaciones_2026_09.csv"
HOJA = 0   # solo para Excel con varias hojas: el nombre entre comillas

original = cargar(ENLACE, HOJA)
print(f"Original: {len(original)} filas y {len(original.columns)} columnas.")
original.head()

Original: 1000 filas y 70 columnas.


,FECHA_PROCESO,NUMERO_SERIE,OFICINA,COD_ADUANA_DESPACHO,ADUANA_DESPACHO,TIPO_IDENT,NIT_EXPORTADOR,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,...,VLR_SERIE_OTROS_GASTOS_USD,COD_ADUANA_SALIDA,ADUANA_SALIDA,FECHA_SOLICITUD_AUTO_EMBARQUE,NUMERO_FORMULARIO,FECHA_DECLARACION_EXPORTACION,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,NIT_DECLARANTE,RAZON_SOCIAL_DECLARANTE
0,202609,1,99,3,Aduanas de Bogotá - Aeropuerto El Dorado,2,830048362,36,0,2,...,0.0,3,Aduanas de Bogotá - Aeropuerto El Dorado,202609,6007784997200,20260909,CENTIGON COLOMBIA S A,CL 25 G 81 B 27,900736525,AGENCIA DE ADUANAS AGEM ADUANA S.A.S NIVEL 2
1,202609,10,99,3,Aduanas de Bogotá - Aeropuerto El Dorado,2,830048362,36,0,2,...,0.0,3,Aduanas de Bogotá - Aeropuerto El Dorado,202609,6007784997218,20260909,CENTIGON COLOMBIA S A,CL 25 G 81 B 27,900736525,AGENCIA DE ADUANAS AGEM ADUANA S.A.S NIVEL 2
2,202609,7,99,3,Aduanas de Bogotá - Aeropuerto El Dorado,2,830048362,36,0,2,...,0.0,3,Aduanas de Bogotá - Aeropuerto El Dorado,202609,6007784997218,20260909,CENTIGON COLOMBIA S A,CL 25 G 81 B 27,900736525,AGENCIA DE ADUANAS AGEM ADUANA S.A.S NIVEL 2
3,202609,6,99,3,Aduanas de Bogotá - Aeropuerto El Dorado,2,830048362,36,0,2,...,0.0,3,Aduanas de Bogotá - Aeropuerto El Dorado,202609,6007784997218,20260909,CENTIGON COLOMBIA S A,CL 25 G 81 B 27,900736525,AGENCIA DE ADUANAS AGEM ADUANA S.A.S NIVEL 2
4,202609,2,99,3,Aduanas de Bogotá - Aeropuerto El Dorado,2,900106777,36,0,2,...,0.0,3,Aduanas de Bogotá - Aeropuerto El Dorado,202609,6007785011636,20260908,SOLDEXEL S.A.S.,BRR BOSQUE TV 54 21 A 120 CENTRO EMPR...,890403077,AGENCIA DE ADUANAS HUBEMAR S.A.S NIVEL 1


## 2. Mira antes de limpiar

El perfil te dice qué revisar: filas idénticas, columnas con vacíos y columnas que deberían ser número o fecha pero salen como texto (`object` o `str`).

Con la papelería sucia: **1 fila idéntica a otra**, `cantidad` y `fecha` salen como texto.

In [23]:
# Celda 3 · Perfil del original
perfil(original)

1000 filas · 70 columnas · filas idénticas a otra: 0


,tipo,vacíos,valores distintos,ejemplo
FECHA_PROCESO,int64,0,1,202609
NUMERO_SERIE,int64,0,68,1
OFICINA,int64,0,1,99
COD_ADUANA_DESPACHO,int64,0,3,3
ADUANA_DESPACHO,object,0,3,Aduanas de Bogotá - Aeropuerto El Dorado
...,...,...,...,...
FECHA_DECLARACION_EXPORTACION,int64,0,28,20260909
RAZON_SOCIAL_EXPORTADOR,object,0,158,CENTIGON COLOMBIA S A
DIREC_EXPORTADOR,object,5,40,CL 25 G 81 B 27
NIT_DECLARANTE,int64,0,40,900736525


In [24]:
# Celda de exploración temporal: analizar valores de ADUANA_DESPACHO
display(original["ADUANA_DESPACHO"].value_counts(dropna=False))

,count
ADUANA_DESPACHO,
Aduanas de Bogotá - Aeropuerto El Dorado,887
Aduanas de Medellín,112
Aduanas de Cali,1


## 3. Empieza (siempre desde el original)

Cada vez que cambies una regla, vuelve a correr **desde aquí** hacia abajo. Si corres una regla dos veces, el cuaderno te avisa.

In [28]:
# Celda 4 · Empezar desde el original
empezar(original)

Empezamos desde el original: 1000 filas.


## 4. Tus reglas, una por celda

Cada regla tiene **nombre**, las **filas** que afecta (True/False), un **motivo** y una **acción**:
- `"quitar"`: salen de la base. Solo para lo que estás seguro de que sobra (una copia exacta).
- `"revisar"`: salen de la base y van al archivo de revisión. Para lo que no sabes cómo arreglar sin inventar (una fecha imposible).

Las reglas de abajo son las de la papelería. **Cámbialas por las tuyas**: usa lo que anotaste en la bitácora el martes.

In [29]:
# Celda 5 · Regla 1: quitar copias exactas
# Una copia exacta es una fila igualita a otra en TODAS las columnas.
regla("Copias exactas",
      base.duplicated(),
      motivo="La misma venta registrada dos veces",
      accion="quitar")

Copias exactas: 0 filas (quitar). Antes 1000, ahora 1000.


,FECHA_PROCESO,NUMERO_SERIE,OFICINA,COD_ADUANA_DESPACHO,ADUANA_DESPACHO,TIPO_IDENT,NIT_EXPORTADOR,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,...,COD_ADUANA_SALIDA,ADUANA_SALIDA,FECHA_SOLICITUD_AUTO_EMBARQUE,NUMERO_FORMULARIO,FECHA_DECLARACION_EXPORTACION,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,NIT_DECLARANTE,RAZON_SOCIAL_DECLARANTE,regla


**Antes de quitar copias, pregúntate:** ¿es el mismo registro dos veces o son dos cosas distintas que se ven iguales (dos ventas iguales el mismo día)? Si tu base **no tiene un código** que identifique cada fila, quizá no debas quitarlas: anótalo como límite.

In [30]:
# Celda 6 · Regla 2: Quitar columnas innecesarias (FECHA_PROCESO)
# Eliminamos la columna FECHA_PROCESO de la base de datos de trabajo
_ya_esta("Quitar FECHA_PROCESO")

# Guardamos registro en la bitácora
antes_columnas = len(base.columns)
base = base.drop(columns=["FECHA_PROCESO"])
bitacora.append({
    "regla": "Quitar FECHA_PROCESO",
    "filas afectadas": 0,
    "acción": "corregir",
    "motivo": "La columna no aporta valor para el análisis de exportaciones",
    "antes": len(base),
    "después": len(base)
})
print(f"Quitar FECHA_PROCESO: columna eliminada. Columnas antes: {antes_columnas}, ahora: {len(base.columns)}.")

Quitar FECHA_PROCESO: columna eliminada. Columnas antes: 70, ahora: 69.


### Fechas y números: primero se convierte, después se decide

`a_fecha` y `a_numero` convierten la columna y te devuelven las filas que **no** se pudieron convertir. Esas filas van a revisión con una regla.

Formatos de fecha comunes (escribe el que tiene tu archivo):

| Así se ve en tu archivo | Formato |
|---|---|
| 2026-08-01 | `"%Y-%m-%d"` |
| 01/08/2026 (día/mes/año) | `"%d/%m/%Y"` |
| 08/01/2026 (mes/día/año) | `"%m/%d/%Y"` |
| 03/22/2019 12:00:00 AM | `"%m/%d/%Y %I:%M:%S %p"` |
| 2/24/2003 0:00 | `"%m/%d/%Y %H:%M"` |
| 2025-01-01T00:00:00.000 | `"%Y-%m-%dT%H:%M:%S.%f"` |

Si **todas** las fechas salen malas, el formato no es ese: míralo otra vez en la celda 2.

In [32]:
# Celda 7 · Regla 3: fechas que no son fechas
# Tomamos el valor original y lo convertimos a un entero limpio antes de pasarlo a texto
valores_originales = original.loc[base.index, "FECHA_DECLARACION_EXPORTACION"]
fecha_texto = pd.to_numeric(valores_originales, errors="coerce").fillna(0).astype(int).astype(str)

# Convertimos al formato de fecha real (AñoMesDía)
fecha_convertida = pd.to_datetime(fecha_texto, format="%Y%m%d", errors="coerce")

# Identificamos si hay filas que no se pudieron convertir y no estaban vacías
malas = fecha_convertida.isna() & valores_originales.notna()
base["FECHA_DECLARACION_EXPORTACION"] = fecha_convertida

print(f"FECHA_DECLARACION_EXPORTACION ahora es fecha. Filas inválidas: {int(malas.sum())}.")

regla("Fecha imposible",
      malas,
      motivo="La fecha no existe o tiene un formato incorrecto",
      accion="revisar")

FECHA_DECLARACION_EXPORTACION ahora es fecha. Filas inválidas detectadas: 0.
Fecha imposible: 0 filas (revisar). Antes 1000, ahora 1000.


,FECHA_PROCESO,NUMERO_SERIE,OFICINA,COD_ADUANA_DESPACHO,ADUANA_DESPACHO,TIPO_IDENT,NIT_EXPORTADOR,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,...,COD_ADUANA_SALIDA,ADUANA_SALIDA,FECHA_SOLICITUD_AUTO_EMBARQUE,NUMERO_FORMULARIO,FECHA_DECLARACION_EXPORTACION,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,NIT_DECLARANTE,RAZON_SOCIAL_DECLARANTE,regla


In [ ]:
# Celda 8 · Regla 4: números que no son números
# Cambia "cantidad" por tu columna. Si tus miles vienen con punto: a_numero("valor", miles=".")
malas = a_numero("cantidad")
regla("Cantidad no numérica",
      malas,
      motivo="La cantidad está en letras; no se adivina",
      accion="revisar")

¿El mismo valor escrito con otra palabra o sin tilde («Bogota» y «Bogotá», «Estrella» y «La Estrella»)? `unificar` no lo ve: usa `reemplazar("columna", {"Bogota": "Bogotá"})`.

¿Necesitas otra regla? Copia una celda de regla, cámbiale el nombre, la condición y el motivo. Ejemplos de condiciones:

```python
base["valor"] < 0                      # valores negativos
base["comuna"].isna()                  # columna vacía
base["fecha"] > "2026-11-24"           # fechas en el futuro
base["edad"] > 120                     # valores imposibles
base.duplicated(subset=["codigo"], keep=False)   # la misma clave con contenido distinto (todas las copias)
```

## 5. ¿Cuadra?

**Original = limpias + quitadas + a revisión.** Si no cuadra, alguna regla perdió o duplicó filas: vuelve a correr desde la celda 4.

Con la papelería sucia: **11 = 8 + 1 + 2** (cuadra)

In [ ]:
# Celda 9 · Conciliar los conteos
conciliar(original)

In [ ]:
# Celda 10 · La bitácora de limpieza (copia las líneas a bitacora.md, sección Limpieza)
ver_bitacora()

In [ ]:
# Celda 11 · Lo que quedó a revisión (no se pierde: queda en su propio archivo)
revision

## 6. Base limpia frente a vista de análisis

La **base limpia** tiene todas las filas válidas. La **vista de análisis** es el pedazo que usa tu pregunta (por ejemplo, solo las ventas completadas). No se borra lo demás: se filtra.

Con la papelería: base limpia **8 filas**; vista de análisis (completadas) **7 filas**, que suman **121.000** pesos (agosto 45.000 y septiembre 76.000).

In [ ]:
# Celda 12 · Vista de análisis: cambia la condición por la de tu pregunta
analisis = base[base["estado"] == "completada"].copy()
print(f"Base limpia: {len(base)} filas · vista de análisis: {len(analisis)} filas")

## 7. Vuelve a correr tus consultas sobre la base limpia

Las consultas de la sesión 3 ahora van sobre `analisis` (o `base`) en lugar de `datos`. ¿Qué cifra cambió frente a la que tenías? Anótalo en la bitácora: **antes · después · por qué**.

In [ ]:
# Celda 13 · Tu cifra de control, ahora con la base limpia (cámbiala por tu consulta)
sql("""
SELECT strftime(fecha, '%Y-%m') AS mes,
       COUNT(*) AS ventas,
       SUM(cantidad * precio_unitario) AS pesos
FROM analisis
GROUP BY mes
ORDER BY mes
""")

## 8. Guarda y sube por PR

1. Corre la celda 14: descarga `…_limpio.csv` y `…_revision.csv`.
2. Menú **Archivo › Descargar › Descargar .ipynb** (este cuaderno, con tus reglas).
3. En GitHub, en una **rama nueva**: los dos CSV a `datos/limpios/` y el cuaderno a `notebooks/02_limpieza.ipynb` (**Add file › Upload files**).
4. Las líneas de la celda 10 a `bitacora.md` (sección **Limpieza**) y, si usaste IA, a **Uso de IA**.
5. Abre el PR: en la descripción, la conciliación (**original = limpias + quitadas + revisión**) y pide revisión a **santiagom26cc**.

**Prueba final:** menú **Entorno de ejecución › Reiniciar y ejecutar todo**. ¿Te da los mismos números? Entonces es reproducible.

In [ ]:
# Celda 14 · Guardar (cambia "ventas" por un nombre corto de tu base)
guardar("ventas")

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.